# Agent Trust Model Comparison

Compare trust evidence from:

- AgentCensus
- A2A Registry

## Notebook setup

Imports, AgentCensus and A2A Registry credentials, and the shared natural-language query live here. Credentials are loaded from environment variables or the ignored `EXPERIMENTS/credentials.yaml` file and are never printed.

In [2]:
import base64
import json
import os
import time
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import quote
from urllib.request import Request, urlopen

try:
    from EXPERIMENTS.utils import (
        AGENTCENSUS_API_BASE,
        a2a_registry_load_api_key,
        a2a_registry_print_table,
        a2a_registry_request_json,
        agentcensus_get_json,
        agentcensus_load_api_key,
        agentcensus_trusted_tls_context,
    )
except ModuleNotFoundError:
    from utils import (
        AGENTCENSUS_API_BASE,
        a2a_registry_load_api_key,
        a2a_registry_print_table,
        a2a_registry_request_json,
        agentcensus_get_json,
        agentcensus_load_api_key,
        agentcensus_trusted_tls_context,
    )

AGENTCENSUS_BEARER_TOKEN = agentcensus_load_api_key(required=True)
A2A_REGISTRY_API_TOKEN = a2a_registry_load_api_key(required=True)

SEARCH_QUERY = (
    "Find an A2A agent that measures AI systems"
)
AGENTCENSUS_SEARCH_LIMIT = 20

ORG_SLUG = os.getenv("AGENTCENSUS_ORG_SLUG")
OWNED_AGENT_KEY = os.getenv("AGENTCENSUS_OWNED_AGENT_KEY")

OUTPUT_DIR = (
    Path("EXPERIMENTS/output")
    if Path("EXPERIMENTS").is_dir()
    else Path("output")
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_json(filename, payload):
    path = OUTPUT_DIR / filename
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    print(path.resolve())
    return path

## Agent search

Assume the caller knows the task but does not already know an agent name, domain, or identifier. Send the same natural-language request to both discovery APIs and keep their complete responses visible.

### Raw AgentCensus search response

`GET /search` performs AgentCensus hybrid retrieval. The complete response is written to `EXPERIMENTS/output/`; the cell prints only its file location. Search rank and match fields describe retrieval relevance, not trust.

In [2]:
agentcensus_search_response = agentcensus_get_json(
    "search",
    {
        "q": SEARCH_QUERY,
        "mechanism": "a2a",
        "limit": AGENTCENSUS_SEARCH_LIMIT,
    },
    token=AGENTCENSUS_BEARER_TOKEN,
)
save_json(
    "trust_model_comparison_agentcensus_search.json",
    agentcensus_search_response,
)

/Users/hanhanwu/Documents/Github/Agentopia/EXPERIMENTS/output/trust_model_comparison_agentcensus_search.json


### Raw A2A Registry search response

`GET /public/agents` receives the same natural-language request. The complete registry response is written to `EXPERIMENTS/output/`; the cell prints only its file location.

In [3]:
a2a_registry_search_response = a2a_registry_request_json(
    "GET",
    "/public/agents",
    params={"q": SEARCH_QUERY, "page": 1},
    authenticated=False,
)
save_json(
    "trust_model_comparison_a2a_registry_search.json",
    a2a_registry_search_response,
)

/Users/hanhanwu/Documents/Github/Agentopia/EXPERIMENTS/output/trust_model_comparison_a2a_registry_search.json


### Three agents retained for the comparison

These are concrete examples from the two search results above:

- **Shared:** Council of AI — Measurement Agent appears in both sources on `councilof.ai`.
- **AgentCensus only:** Selnoviktech Agent appears in the AgentCensus results and was not returned by the A2A Registry search above.
- **A2A Registry only:** A2A402 Agent Marketplace appears in the registry results and was not returned by the AgentCensus search above; the later AgentCensus domain read also returns `404 not_found`.

The labels describe what these API checks returned, not proof that an agent does or does not exist elsewhere.

In [4]:
AGENTS = [
    {
        "case": "shared",
        "output_label": "shared_council_of_ai",
        "name": "Council of AI — Measurement Agent",
        "domain": "councilof.ai",
        "agentcensus_agent_key": "ag_0257be5ab061",
        "a2a_registry_package": "ai.councilof.council_of_ai__measurement_agent",
        "a2a_manifest_url": "https://councilof.ai/.well-known/agent.json",
    },
    {
        "case": "agentcensus_only",
        "output_label": "only_selnoviktech",
        "name": "Selnoviktech Agent",
        "domain": "selnoviktech.com",
        "agentcensus_agent_key": "ag_050fb7a9ceea",
        "a2a_registry_package": None,
        "a2a_manifest_url": None,
    },
    {
        "case": "a2a_registry_only",
        "output_label": "missing_a2a402",
        "name": "A2A402 Agent Marketplace",
        "domain": "a2a402.market",
        "agentcensus_agent_key": None,
        "a2a_registry_package": "market.a2a402.a2a402_agent_origin_market",
        "a2a_manifest_url": "https://a2a402.market/.well-known/agent-card.json",
    },
]

a2a_registry_print_table(
    AGENTS,
    [
        "case",
        "name",
        "domain",
        "agentcensus_agent_key",
        "a2a_registry_package",
        "a2a_manifest_url",
    ],
    limit=len(AGENTS),
)

known_agentcensus_keys = {
    agent["agentcensus_agent_key"]
    for agent in AGENTS
    if agent["agentcensus_agent_key"]
}
if OWNED_AGENT_KEY and OWNED_AGENT_KEY not in known_agentcensus_keys:
    raise ValueError("AGENTCENSUS_OWNED_AGENT_KEY is not one of the selected agents.")

case              | name                              | domain           | agentcensus_agent_key | a2a_registry_package                          | a2a_manifest_url                                 
------------------+-----------------------------------+------------------+-----------------------+-----------------------------------------------+--------------------------------------------------
shared            | Council of AI — Measurement Agent | councilof.ai     | ag_0257be5ab061       | ai.councilof.council_of_ai__measurement_agent | https://councilof.ai/.well-known/agent.json      
agentcensus_only  | Selnoviktech Agent                | selnoviktech.com | ag_050fb7a9ceea       | None                                          | None                                             
a2a_registry_only | A2A402 Agent Marketplace          | a2a402.market    | None                  | market.a2a402.a2a402_agent_origin_market      | https://a2a402.market/.well-known/agent-card.json


## Identity & Control

Identity and control asks **who operates an agent** and **whether control has been proved**.

The public agent and domain records are identity evidence, but they do not by themselves prove a real-world organization controls the agent or domain. Agent Control Verification (ACV) claims and domain claims are stronger control evidence and are only available inside the organization that owns them.

### Raw public agent record

`GET /agents/{agentKey}` returns the published identity, discovery provenance, observed status, and related raw fields for one census record. Each complete response is written to `EXPERIMENTS/output/`; the cell prints only the file locations.

In [5]:
agent_identity_responses = {}
for agent in AGENTS:
    agent_key = agent["agentcensus_agent_key"]
    if agent_key is None:
        continue
    agent_identity_responses[agent_key] = agentcensus_get_json(
        f"agents/{agent_key}", token=AGENTCENSUS_BEARER_TOKEN
    )
    save_json(
        f"trust_model_comparison_agentcensus_{agent['output_label']}_agent.json",
        agent_identity_responses[agent_key],
    )

/Users/hanhanwu/Documents/Github/Agentopia/EXPERIMENTS/output/trust_model_comparison_agentcensus_shared_council_of_ai_agent.json
/Users/hanhanwu/Documents/Github/Agentopia/EXPERIMENTS/output/trust_model_comparison_agentcensus_only_selnoviktech_agent.json


### Raw public domain record

`GET /domains/{domain}` returns the domain-level record and observations. Each complete response is written to `EXPERIMENTS/output/`; the cell prints only the file locations. A domain record is not, by itself, proof that the agent is controlled by a named organization.

In [6]:
domain_identity_responses = {}
for agent in AGENTS:
    domain = agent["domain"]
    domain_identity_responses[domain] = agentcensus_get_json(
        f"domains/{domain}", token=AGENTCENSUS_BEARER_TOKEN
    )
    save_json(
        f"trust_model_comparison_agentcensus_{agent['output_label']}_domain.json",
        domain_identity_responses[domain],
    )

/Users/hanhanwu/Documents/Github/Agentopia/EXPERIMENTS/output/trust_model_comparison_agentcensus_shared_council_of_ai_domain.json
/Users/hanhanwu/Documents/Github/Agentopia/EXPERIMENTS/output/trust_model_comparison_agentcensus_only_selnoviktech_domain.json
/Users/hanhanwu/Documents/Github/Agentopia/EXPERIMENTS/output/trust_model_comparison_agentcensus_missing_a2a402_domain.json


### Raw organization domain claims

`GET /orgs/{slug}/domains` lists this organization's domain-control claims, including their raw states and challenge details. The response is written to `EXPERIMENTS/output/`, and the cell prints only its file location. The request runs only when `AGENTCENSUS_ORG_SLUG` is set. It does not start or verify a claim.

In [7]:
if ORG_SLUG:
    domain_claims_response = agentcensus_get_json(
        f"orgs/{ORG_SLUG}/domains", token=AGENTCENSUS_BEARER_TOKEN
    )
    save_json(
        f"trust_model_comparison_agentcensus_{ORG_SLUG}_domain_claims.json",
        domain_claims_response,
    )

### Raw agent-control claim or permitted-method offer

`GET /orgs/{slug}/agents/{agentKey}/claim` returns either the organization's existing ACV claim or the control-verification methods permitted for that agent. The response is written to `EXPERIMENTS/output/`, and the cell prints only its file location. The request runs only for the configured organization and owned agent.

In [8]:
if ORG_SLUG and OWNED_AGENT_KEY:
    agent_claim_response = agentcensus_get_json(
        f"orgs/{ORG_SLUG}/agents/{OWNED_AGENT_KEY}/claim",
        token=AGENTCENSUS_BEARER_TOKEN,
    )
    save_json(
        f"trust_model_comparison_agentcensus_{OWNED_AGENT_KEY}_claim.json",
        agent_claim_response,
    )

### Raw signed ACV assertion

`GET /orgs/{slug}/agents/{agentKey}/claim/assertion` returns binary COSE_Sign1 data when a verified claim has an assertion. The status, safe headers, and complete body encoded as base64 are written to a JSON file in `EXPERIMENTS/output/`; the cell prints only its location. A `404` response is also preserved as raw evidence that no assertion is currently available.

In [9]:
if ORG_SLUG and OWNED_AGENT_KEY:
    assertion_url = (
        f"{AGENTCENSUS_API_BASE}/orgs/{quote(ORG_SLUG, safe='')}"
        f"/agents/{quote(OWNED_AGENT_KEY, safe='')}/claim/assertion"
    )
    assertion_request = Request(
        assertion_url,
        headers={
            "Accept": "application/cose",
            "Authorization": f"Bearer {AGENTCENSUS_BEARER_TOKEN}",
            "User-Agent": "agentopia-trust-model-comparison/0.1",
        },
        method="GET",
    )
    assertion_started = time.perf_counter()
    try:
        with urlopen(
            assertion_request, timeout=30, context=agentcensus_trusted_tls_context()
        ) as assertion_http_response:
            assertion_status = assertion_http_response.status
            assertion_headers = dict(assertion_http_response.headers.items())
            assertion_body = assertion_http_response.read()
    except HTTPError as error:
        assertion_status = error.code
        assertion_headers = dict(error.headers.items())
        assertion_body = error.read()
    except URLError as error:
        assertion_status = None
        assertion_headers = {}
        assertion_body = str(error.reason).encode("utf-8")

    assertion_elapsed_ms = round((time.perf_counter() - assertion_started) * 1000, 1)
    safe_assertion_headers = {
        name: value
        for name, value in assertion_headers.items()
        if name.lower() in {
            "content-type", "content-length", "x-request-id",
            "x-ratelimit-limit", "x-ratelimit-remaining", "x-ratelimit-reset",
        }
    }
    assertion_output = {
        "status": assertion_status,
        "url": assertion_url,
        "elapsedMs": assertion_elapsed_ms,
        "headers": safe_assertion_headers,
        "bodyBase64": base64.b64encode(assertion_body).decode("ascii"),
    }
    save_json(
        f"trust_model_comparison_agentcensus_{OWNED_AGENT_KEY}_claim_assertion.json",
        assertion_output,
    )

### Write boundary

This notebook does not call the following state-changing endpoints:

- `POST /orgs/{slug}/agents/{agentKey}/claim`
- `POST /orgs/{slug}/agents/{agentKey}/claim/verify`
- `POST /orgs/{slug}/domains`
- `POST /orgs/{slug}/domains/{domain}/verify`

They require an owned agent or domain and explicit approval before use.